# ExDark: Zero-DCE++ (weight Hiền Anh) → YOLOv8n

Pipeline của notebook:

`ExDark tối → Zero-DCE++ pretrained → YOLOv8n → đánh giá test`

- Dùng module `src/hienanh/enhancement.py`.
- Cách lấy weight theo notebook `Notebooks/Tiep/ExDark_ZeroDCEpp_Bilateral_YOLOv8_Kaggle.ipynb`.
- Dùng checkpoint `Results/hienanh/weights/zerodcepp_Epoch99.pth` do `src/hienanh/download_weights.py` tải và kiểm tra SHA-256. Notebook vẫn hỗ trợ đường dẫn cũ trong `src/hienanh`.
- Zero-DCE++ chạy với `scale_factor=4`, không fine-tune trên ExDark.
- Notebook này không áp dụng CLAHE hoặc bilateral denoise.
- Kaggle cần bật **GPU** và **Internet** trước khi Run All.

In [ ]:
!pip install -q -U ultralytics opencv-python-headless pyyaml tqdm matplotlib pandas

from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import torch
import yaml
from tqdm.auto import tqdm

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. Cấu hình thí nghiệm

In [ ]:
MODEL_NAME = 'yolov8n.pt'
EPOCHS = 40
IMAGE_SIZE = 640
BATCH_SIZE = 16
PATIENCE = 15
SEED = 42

ZERO_VARIANT = 'zerodcepp'
ZERO_SCALE_FACTOR = 4
EXPECTED_SHA256 = 'ca8855b90df9a80fa4195a831f33d3476b1964f787eb70602797c773067f3b84'

WORK = Path('/kaggle/working')
REPO = WORK / 'Low-Light-Image-Enhancement-and-Downstream-Recognition'
PROJECT = REPO
REPO_URL = 'https://github.com/NMH1203/Low-Light-Image-Enhancement-and-Downstream-Recognition.git'
RAW = REPO / 'Dataset' / 'exdark_yolo_dark'
ENHANCED = WORK / 'exdark_yolo_zerodcepp_hienanh'
RUNS = WORK / 'runs_zerodcepp_hienanh'
RUN_NAME = 'zerodcepp_hienanh_yolov8n_e40'

CONFIG = {
    'model': MODEL_NAME, 'epochs': EPOCHS, 'imgsz': IMAGE_SIZE,
    'batch': BATCH_SIZE, 'patience': PATIENCE, 'seed': SEED,
    'enhancer': ZERO_VARIANT, 'scale_factor': ZERO_SCALE_FACTOR,
}
print(json.dumps(CONFIG, indent=2))

## 2. Lấy mã nguồn và weight Zero-DCE++

Dùng cùng checkpoint Zero-DCE++ được tham chiếu trong notebook của Tiệp. Script của Hiền Anh tải checkpoint chính thức; notebook kiểm tra SHA-256 thêm một lần trước khi suy luận.

In [ ]:
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)

assert RAW.is_dir(), f'Không tìm thấy dataset: {RAW}'
WEIGHT_CANDIDATES = [
    # Đường dẫn thực tế được download_weights.py in ra trên Kaggle.
    PROJECT / 'Results/hienanh/weights/zerodcepp_Epoch99.pth',
    # Đường dẫn trong phiên bản cũ của notebook Tiệp.
    PROJECT / 'src/hienanh/Results/weights/zerodcepp_Epoch99.pth',
]
WEIGHTS = next((path for path in WEIGHT_CANDIDATES if path.is_file()), None)
if WEIGHTS is None:
    print('Chưa có weights trong Kaggle; tải bằng script của Hiền Anh...')
    subprocess.run(
        [sys.executable, 'src/hienanh/download_weights.py'],
        cwd=PROJECT, check=True,
    )
    WEIGHTS = next((path for path in WEIGHT_CANDIDATES if path.is_file()), None)

assert WEIGHTS is not None, (
    'Không tìm thấy zerodcepp_Epoch99.pth tại: '
    + ', '.join(str(path) for path in WEIGHT_CANDIDATES)
)
sha = hashlib.sha256()
with WEIGHTS.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b''):
        sha.update(chunk)
weight_hash = sha.hexdigest()
assert weight_hash == EXPECTED_SHA256, (
    'Checksum sai: checkpoint hỏng hoặc không đúng bản chính thức.'
)
print('Weight hợp lệ:', WEIGHTS)
print('SHA-256:', weight_hash)

## 3. Nạp Zero-DCE++ theo code Hiền Anh

In [ ]:
sys.path.insert(0, str(REPO))
from src.hienanh.enhancement import enhance_neural, load_model

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
zero_model = load_model(
    WEIGHTS, variant=ZERO_VARIANT, scale_factor=ZERO_SCALE_FACTOR, device=DEVICE
)
parameter_count = sum(p.numel() for p in zero_model.parameters())
print('Thiết bị:', DEVICE)
print('Số tham số Zero-DCE++:', f'{parameter_count:,}')

## 4. Tạo dataset Zero-DCE++

Split và nhãn YOLO được giữ nguyên; chỉ pixel ảnh thay đổi. Ảnh kết quả được lưu JPEG chất lượng 95 để dung lượng phù hợp với Kaggle.

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
SPLITS = {'train': 'train', 'val': 'valid', 'test': 'test'}

def read_yaml(root):
    with open(root / 'data.yaml', encoding='utf-8-sig') as f:
        return yaml.safe_load(f)

def locate_images(root, config, split):
    aliases = ('val', 'valid') if split == 'val' else (split,)
    candidates = []
    configured = config.get(split)
    if isinstance(configured, str):
        p = Path(configured)
        candidates.append(p if p.is_absolute() else root / p)
    for alias in aliases:
        candidates.extend([root / alias / 'images', root / 'images' / alias])
    found = next((p for p in candidates if p.is_dir()), None)
    if found is None:
        raise FileNotFoundError(f'Không tìm thấy split {split} trong {root}')
    return found

def locate_labels(root, image_dir):
    parts = list(image_dir.relative_to(root).parts)
    parts[parts.index('images')] = 'labels'
    label_dir = root.joinpath(*parts)
    if not label_dir.is_dir():
        raise FileNotFoundError(label_dir)
    return label_dir

def build_zerodcepp_dataset(source, output):
    if output.exists():
        shutil.rmtree(output)
    source_yaml = read_yaml(source)
    counts, elapsed = {}, {}
    for split, out_split in SPLITS.items():
        image_dir = locate_images(source, source_yaml, split)
        label_dir = locate_labels(source, image_dir)
        out_images = output / out_split / 'images'
        out_labels = output / out_split / 'labels'
        out_images.mkdir(parents=True)
        out_labels.mkdir(parents=True)
        files = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
        started = time.perf_counter()
        for image_path in tqdm(files, desc=f'Zero-DCE++ {split}'):
            image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
            if image is None:
                raise ValueError(f'Không đọc được {image_path}')
            label_path = label_dir / f'{image_path.stem}.txt'
            if not label_path.is_file():
                raise FileNotFoundError(label_path)
            enhanced = enhance_neural(image, zero_model, device=DEVICE)
            out_name = f'{image_path.stem}.jpg'
            ok = cv2.imwrite(str(out_images / out_name), enhanced, [cv2.IMWRITE_JPEG_QUALITY, 95])
            if not ok:
                raise RuntimeError(f'Không ghi được {out_name}')
            shutil.copy2(label_path, out_labels / f'{image_path.stem}.txt')
        counts[split] = len(files)
        elapsed[split] = time.perf_counter() - started

    output_yaml = {
        'path': str(output.resolve()), 'train': 'train/images',
        'val': 'valid/images', 'test': 'test/images', 'names': source_yaml['names']
    }
    with open(output / 'data.yaml', 'w', encoding='utf-8') as f:
        yaml.safe_dump(output_yaml, f, sort_keys=False, allow_unicode=True)
    metadata = {
        'method': 'Zero-DCE++ pretrained via src/hienanh',
        'variant': ZERO_VARIANT, 'scale_factor': ZERO_SCALE_FACTOR,
        'checkpoint': str(WEIGHTS), 'checkpoint_sha256': weight_hash,
        'fine_tuned_on_exdark': False, 'counts': counts, 'elapsed_seconds': elapsed,
    }
    (output / 'enhancement_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    return metadata

enhancement_metadata = build_zerodcepp_dataset(RAW, ENHANCED)
print(json.dumps(enhancement_metadata, indent=2))

## 5. Kiểm tra ảnh trước và sau tăng sáng

In [ ]:
raw_yaml = read_yaml(RAW)
raw_val = locate_images(RAW, raw_yaml, 'val')
sample = sorted(p for p in raw_val.iterdir() if p.suffix.lower() in IMAGE_EXTS)[0]
raw_image = cv2.imread(str(sample))
enhanced_image = cv2.imread(str(ENHANCED / 'valid' / 'images' / f'{sample.stem}.jpg'))

def stats(image):
    lab_l = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)[:, :, 0]
    return f'L mean={lab_l.mean():.1f}, L std={lab_l.std():.1f}'

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, title, image in zip(axes, ['Ảnh tối gốc', 'Zero-DCE++ (Hiền Anh)'], [raw_image, enhanced_image]):
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{title}\n{stats(image)}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Train YOLOv8n

In [ ]:
from ultralytics import YOLO

YOLO_DEVICE = 0 if torch.cuda.is_available() else 'cpu'
detector = YOLO(MODEL_NAME)
train_results = detector.train(
    data=str(ENHANCED / 'data.yaml'),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    seed=SEED,
    deterministic=True,
    device=YOLO_DEVICE,
    project=str(RUNS),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
)

## 7. Đánh giá trên tập test và xuất số liệu

In [ ]:
BEST = RUNS / RUN_NAME / 'weights' / 'best.pt'
assert BEST.is_file(), f'Không tìm thấy checkpoint: {BEST}'
best_model = YOLO(str(BEST))
test_metrics = best_model.val(
    data=str(ENHANCED / 'data.yaml'),
    split='test', imgsz=IMAGE_SIZE, batch=BATCH_SIZE, device=YOLO_DEVICE,
    project=str(RUNS), name='zerodcepp_hienanh_test', plots=True,
)

summary = {
    'experiment': 'Zero-DCE++ Hiền Anh + YOLOv8n',
    **CONFIG,
    'precision': float(test_metrics.box.mp),
    'recall': float(test_metrics.box.mr),
    'mAP50': float(test_metrics.box.map50),
    'mAP50_95': float(test_metrics.box.map),
    'best_weights': str(BEST),
}
SUMMARY_JSON = WORK / 'zerodcepp_hienanh_yolov8n_results.json'
SUMMARY_CSV = WORK / 'zerodcepp_hienanh_yolov8n_results.csv'
SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding='utf-8')
pd.DataFrame([summary]).to_csv(SUMMARY_CSV, index=False)
display(pd.DataFrame([summary]))

## 8. Xem dự đoán mẫu

In [ ]:
test_images = sorted((ENHANCED / 'test' / 'images').glob('*.jpg'))[:6]
predictions = best_model.predict(test_images, imgsz=IMAGE_SIZE, conf=0.25, device=YOLO_DEVICE, verbose=False)
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, prediction in zip(axes.ravel(), predictions):
    plotted = prediction.plot()
    ax.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
    ax.set_title(Path(prediction.path).name)
    ax.axis('off')
for ax in axes.ravel()[len(predictions):]:
    ax.axis('off')
plt.tight_layout()
plt.show()
print('YOLO best.pt:', BEST)
print('Kết quả JSON:', SUMMARY_JSON)
print('Kết quả CSV:', SUMMARY_CSV)

## 9. Đóng gói và tải kết quả về máy

File ZIP gồm checkpoint YOLO, kết quả train/test, biểu đồ, cấu hình và bảng chỉ số. Dataset Zero-DCE++ không được đóng gói để tránh file tải xuống quá lớn.

In [ ]:
from IPython.display import FileLink, display

TRAIN_RUN = RUNS / RUN_NAME
TEST_RUN = RUNS / 'zerodcepp_hienanh_test'
EXPORT_DIR = WORK / 'zerodcepp_hienanh_yolov8n_export'
ARCHIVE = WORK / 'zerodcepp_hienanh_yolov8n_results.zip'

required_files = [
    BEST, SUMMARY_JSON, SUMMARY_CSV,
    ENHANCED / 'enhancement_metadata.json',
    ENHANCED / 'data.yaml',
]
missing = [path for path in required_files if not path.is_file()]
assert not missing, 'Thiếu kết quả: ' + ', '.join(str(path) for path in missing)

if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir(parents=True)

# Sao chép toàn bộ log, biểu đồ và weights của lần train/test.
shutil.copytree(TRAIN_RUN, EXPORT_DIR / 'train', dirs_exist_ok=True)
if TEST_RUN.is_dir():
    shutil.copytree(TEST_RUN, EXPORT_DIR / 'test', dirs_exist_ok=True)

for path in required_files[1:]:
    shutil.copy2(path, EXPORT_DIR / path.name)

if ARCHIVE.exists():
    ARCHIVE.unlink()
created_zip = Path(shutil.make_archive(
    str(ARCHIVE.with_suffix('')), 'zip', root_dir=EXPORT_DIR
))
size_mb = created_zip.stat().st_size / (1024 ** 2)
print(f'Đã tạo: {created_zip} ({size_mb:.1f} MB)')
display(FileLink(str(created_zip), result_html_prefix='Bấm vào đây để tải kết quả: '))